In [1]:
import torch
from torch import nn
from d2l import torch as d2l

def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i+h, j:j+w] * K).sum()
    return Y

In [2]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

In [4]:
conv2d = nn.Conv2d(1, 1, kernel_size=(1, 2), bias=False)

X = torch.ones((6, 8))
X[:, 2:6] = 0
K = torch.tensor([[1.0, -1.0]])
Y = corr2d(X, K)
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))

for i in range(10):
    Y_hat = conv2d(X)
    l = ((Y_hat - Y) ** 2).sum()
    conv2d.zero_grad()
    l.backward()
    conv2d.weight.data[:] -= 3e-2 * conv2d.weight.grad
    print(f'batch {i+1}, loss {l:.3f}')

batch 1, loss 25.627
batch 2, loss 10.569
batch 3, loss 4.376
batch 4, loss 1.822
batch 5, loss 0.765
batch 6, loss 0.326
batch 7, loss 0.141
batch 8, loss 0.063
batch 9, loss 0.029
batch 10, loss 0.014


In [5]:
conv2d.weight

Parameter containing:
tensor([[[[ 0.9755, -0.9910]]]], requires_grad=True)